In [49]:
from __future__ import annotations
from pathlib import Path

In [50]:
OBO_PATH = Path("data/go-basic.obo")
NAMESPACE_SUFFIX = {
    "molecular_function": "mf",
    "biological_process": "bp",
    "cellular_component": "cc",
}

In [51]:
def load_namespaces(obo_path: Path) -> dict[str, str]:
    namespaces: dict[str, str] = {}
    current_id: str | None = None
    current_namespace: str | None = None
    current_alt_ids: list[str] = []

    def flush_term() -> None:
        nonlocal current_id, current_namespace, current_alt_ids
        if current_id and current_namespace:
            namespaces[current_id] = current_namespace
            for alt_id in current_alt_ids:
                namespaces[alt_id] = current_namespace
        current_id = None
        current_namespace = None
        current_alt_ids = []

    with obo_path.open() as handle:
        for raw_line in handle:
            line = raw_line.rstrip("\n")
            if line == "[Term]":
                flush_term()
                continue
            if not line:
                flush_term()
                continue
            if line.startswith("id: "):
                current_id = line[4:]
            elif line.startswith("namespace: "):
                current_namespace = line[11:]
            elif line.startswith("alt_id: "):
                current_alt_ids.append(line[8:])

    flush_term()
    return namespaces




In [52]:
def split_tsv(tsv_path: Path, namespaces: dict[str, str]) -> dict[str, object]:
    output_paths = {
        namespace: tsv_path.with_name(f"{tsv_path.stem}_{suffix}.tsv")
        for namespace, suffix in NAMESPACE_SUFFIX.items()
    }
    counts = {suffix: 0 for suffix in NAMESPACE_SUFFIX.values()}
    missing_counts: dict[str, int] = {}
    total_rows = 0
    missing_rows = 0
    handles = {namespace: path.open("w") for namespace, path in output_paths.items()}

    try:
        with tsv_path.open() as source:
            for line_no, raw_line in enumerate(source, start=1):
                line = raw_line.rstrip("\n")
                if not line:
                    continue

                parts = line.split("\t")
                if len(parts) < 2:
                    raise ValueError(f"{tsv_path}:{line_no} does not have at least 2 columns")

                go_id = parts[1]
                total_rows += 1
                namespace = namespaces.get(go_id)
                if namespace is None:
                    missing_rows += 1
                    missing_counts[go_id] = missing_counts.get(go_id, 0) + 1
                    continue

                handles[namespace].write(raw_line)
                counts[NAMESPACE_SUFFIX[namespace]] += 1
    finally:
        for handle in handles.values():
            handle.close()

    return {
        "total_rows": total_rows,
        "counts": counts,
        "missing_rows": missing_rows,
        "missing_counts": missing_counts,
        "output_paths": output_paths,
    }

In [ ]:
def split(path_to_tsv: str) -> None:
    tsv_path = Path(path_to_tsv)
    namespaces = load_namespaces(OBO_PATH)
    result = split_tsv(tsv_path, namespaces)

    print(f"source_file={tsv_path}")
    print(f"total_rows={result['total_rows']}")
    print(f"mf_rows={result['counts']['mf']}")
    print(f"bp_rows={result['counts']['bp']}")
    print(f"cc_rows={result['counts']['cc']}")
    print(f"missing_rows={result['missing_rows']}")

    if result['missing_counts']:
        print("missing_go_terms=")
        for go_id, count in sorted(result['missing_counts'].items()):
            print(f"  {go_id}: {count}")
    else:
        print("missing_go_terms=none")

In [ ]:
split("data/pe3.tsv")

source_file=data/pe3.tsv
total_rows=5984
mf_rows=1876
bp_rows=1261
cc_rows=2847
missing_rows=0
missing_go_terms=none


In [ ]:
split("data/sprof.tsv")


source_file=data/sprof.tsv
total_rows=109680
mf_rows=36560
bp_rows=36560
cc_rows=36560
missing_rows=0
missing_go_terms=none
